# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# The metadata object provides access to dataset metadata as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

### Step 1: List all Record Sets
In Croissant, datasets are organized into *record sets*. Each record set describes a data table (often corresponding to a CSV or similar resource) and is uniquely identified by its `@id`.

We'll list all record sets and print their `@id` and name.

In [ ]:
# List all record sets with their @id and name

def get_record_sets(meta):
    # Croissant schemas put record sets under the 'recordSet' attribute.
    if hasattr(meta, 'recordSet'):
        return meta.recordSet
    return []

record_sets = get_record_sets(dataset.metadata)

print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name', 'N/A')}")
    print(f"    description: {rs.get('description', 'N/A')}\n")
    record_set_ids.append(rs['@id'])

if not record_sets:
    print('No record sets listed directly in metadata. Attempting to list available records via the mlcroissant Dataset...')

### Step 2: Explore the Records API
If no record sets are present under `metadata.recordSet`, we can attempt to discover available record sets by using the dataset interface. We'll use `dataset.record_sets()`.

In [ ]:
# Discover record sets using the mlcroissant API
record_set_names = dataset.record_sets()
print(f"Available record sets found in the schema: {record_set_names}")

### Step 3: List Fields and Columns for a Record Set
Let’s look at the first available record set (by `@id`). We'll enumerate all available fields and their `@id`.

In [ ]:
# Use the first discovered record set
selected_record_set = record_set_names[0]
print(f"\nExploring record set: {selected_record_set}\n")

# List all fields and their @id
metadata_json = dataset.metadata.to_jsonld()

# Function to find the record set object in the raw metadata
def find_record_set_obj(meta_jsonld, record_set_id):
    # Look for recordSet element(s) at the root
    rs_list = meta_jsonld.get('recordSet', [])
    for rs in rs_list:
        if isinstance(rs, dict) and rs.get('@id') == record_set_id:
            return rs
    # Some schemas may lack top-level recordSet, try @graph
    for obj in meta_jsonld.get('@graph', []):
        if obj.get('@id') == record_set_id:
            return obj
    return None

record_set_obj = find_record_set_obj(metadata_json, selected_record_set)

if record_set_obj and 'field' in record_set_obj:
    print(f"Fields in record set '{selected_record_set}':\n")
    for f in record_set_obj['field']:
        if isinstance(f, dict):
            print(f"  - @id: {f.get('@id')}  name: {f.get('name', 'N/A')}")
        else:
            print(f"  - {f}")
else:
    print("Could not find field definitions in the selected record set; consider inspecting records directly.")

### Step 4: Preview a Few Records
Preview a few records from the selected record set, which helps in understanding the available columns/fields and their structure.

In [ ]:
# Print a few records using the high-level interface:
print(f"Sample records from record set '{selected_record_set}':\n")
for i, rec in enumerate(dataset.records(record_set=selected_record_set)):
    print(json.dumps(rec, indent=2))
    if i >= 2:
        break

## 3. Data Extraction
Load data from the main record set into a pandas DataFrame for analysis. Use the record set and field `@id`s.

In [ ]:
dataframes = {}

# We'll operate on the main record set at selected_record_set
print(f"Loading all records from: {selected_record_set}")
records = list(dataset.records(record_set=selected_record_set))
df = pd.DataFrame(records)
dataframes[selected_record_set] = df
print(f"Loaded DataFrame columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering, normalization, and grouping.

**Note:** For demonstration, identify one numeric field and one possible grouping field using the auto-detected columns.

In [ ]:
# List the columns to help select fields
df = dataframes[selected_record_set]
print("DataFrame columns:", df.columns.tolist())

# Guess typical protein dataset columns for EDA
# You may need to adjust these column names based on above printout
possible_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ["count", "abundance", "coverage", "mw", "intensity", "number"])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # fallback: try first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    else:
        numeric_field = df.columns[0]

# Try grouping by a categorical field, e.g. 'Sample', 'description', etc.
possible_group_fields = [col for col in df.columns if any(s in col.lower() for s in ["sample", "description", "group", "category"])]
group_field = possible_group_fields[0] if possible_group_fields else None

print(f"Using numeric_field='{numeric_field}' and group_field='{group_field}' (if available).")

# Remove missing values for the numeric field
eda_df = df.copy()
eda_df = eda_df[pd.to_numeric(eda_df[numeric_field], errors='coerce').notnull()]
eda_df[numeric_field] = pd.to_numeric(eda_df[numeric_field], errors='coerce')

# Example filter: Show records with values above a threshold
threshold = eda_df[numeric_field].mean() if eda_df.shape[0]>0 else 0
filtered_df = eda_df[eda_df[numeric_field] > threshold]
print(f"\nFiltered records where {numeric_field} > mean ({threshold:.2f}):")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' values:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Optionally group
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nMean {numeric_field} by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and (if available) the distribution by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=30, kde=True, color='teal')
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

# Boxplot by group, if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR^2 dataset metadata and records using `mlcroissant`, referencing record sets and fields by `@id`.
- Explored available record sets and their fields.
- Loaded a main record set into a pandas DataFrame for analysis.
- Filtered, normalized, and grouped data for basic exploratory analysis.
- Visualized distributions in the data.

This workflow can be adapted to other Croissant-compliant datasets for FAIR data exploration.

*Make sure to cite the dataset if you use it:*
> Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.